Imports

In [ ]:
#Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from pathlib import Path
import sys

#Root
project_root = Path.cwd().parent
sys.path.append(str(project_root))

#Utils
from src.utils.db import get_engine
from src.utils.logger import setup_logger

logger = setup_logger(__name__)
engine = get_engine()

logger.info("Environment ready")

In [ ]:
#Avalaible tables

query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name
"""

tables = pd.read_sql(query,engine)
print(tables)

In [ ]:
# Rows per table
tables_info = []

for table in ['holidays_events', 'items', 'oil', 'stores', 'train', 'transactions']:
    query = f"SELECT COUNT(*) as count FROM {table};"
    count = pd.read_sql(query, engine).iloc[0, 0]
    tables_info.append({'table': table, 'rows': count})

df_tables = pd.DataFrame(tables_info)

print("\nRows per table:")
print(df_tables)

In [ ]:
#Stores table
stores = pd.read_sql("SELECT * FROM stores",engine)
stores.head()
stores = stores.sort_values('cluster', ascending = False)
stores_state = stores['state'].value_counts()

stores_cities = stores['city'].value_counts()

print("Stores summary:")
print(f"\n Total stores: {len(stores)}")
print(f"\n Number of cities with stores: {stores['city'].nunique()}")
print(f"\n Number of states with stores: {stores['state'].nunique()}")
print(f"\n Number of types of stores: {stores['type'].nunique()}")
print(f"\n Number of clusters: {stores['cluster'].nunique()}")
print(f"\nTotal of missing values: \n{stores.isna().sum()}")




fig = make_subplots(rows = 3, cols= 1 , subplot_titles = ('Total stores per State','Total Stores per City','Total Stores by Type and Cluster'))



fig.add_trace(
    go.Bar(x = stores_state.index, y = stores_state.values, text = stores_state.values),
    col = 1, row = 1
)

fig.add_trace(
    go.Bar(x = stores_cities.index, y = stores_cities.values, text = stores_cities.values),
    col = 1, row = 2
)

fig_temp = px.bar(stores, 'type',color='cluster')
for trace in fig_temp.data:
    fig.add_trace(trace, row=3, col=1)


fig.update_layout(height = 1200, coloraxis_showscale=False, showlegend = False,
                  xaxis_title='State',yaxis_title ='Total Stores',
                  xaxis2_title='City',yaxis2_title ='Total Stores',
                  xaxis3_title='Type/Cluster',yaxis3_title ='Total Stores'
                  )


In [ ]:
# Items

items = pd.read_sql("SELECT * FROM items;", engine)

print("Items summary:")
print(f"\nTotal items: {len(items)}")
print(f"\nTotal of family items: {items['family'].nunique()}")
print(f"\nTotal of classes: {items['class'].nunique()}")
print(f"\nTotal of missing values: \n{items.isna().sum()}")


top_families = items['family'].value_counts().head(10)
amount_perishable = items['perishable'].value_counts()

amount_perishable.index = ['Perishable', 'Non-Perishable']

fig = make_subplots(rows = 1, cols = 2, subplot_titles = ('Top 10 Items Families','Total Perishable and Non-Perishable Items'))

fig.add_trace(
    go.Bar(x = top_families.index , y = top_families.values, text = top_families.values),
    col = 1, row = 1 
)

fig.add_trace(
    go.Bar(x = amount_perishable.index , y = amount_perishable.values, text = amount_perishable.values),
    col = 2, row = 1 
)



fig.update_layout(showlegend=False, height=400,
                  xaxis_title='Family',yaxis_title ='Total items',
                  xaxis2_title='Perishability',yaxis2_title ='Total items',

                  )
fig.update_traces(texttemplate='%{text:,.0f}')

fig.show()

In [ ]:
#Oil table

oil = pd.read_sql("SELECT * FROM oil", engine)

print("Oil summary:")
print(f"\nTotal days: {len(oil)}")
print(f"\nTotal missing values: \n{oil.isna().sum()}")


fig = px.line(oil,'date','dcoilwtico',title ="Historical oil cost")

fig.update_layout(xaxis_title='Date',yaxis_title='Oil Cost')

fig.show()

In [ ]:
#Holidays table
query = """
        SELECT 
                *, 
                EXTRACT (YEAR FROM date) as year
        FROM holidays_events;
        """
holidays = pd.read_sql(query, engine)

display(holidays.head())
print("Holidays summary:")
print(f"\nTotal of holidays: {len(holidays)}")
print(f"\nTotal of Holidays types: {holidays['type'].nunique()}")
print(f"\nTotal of missing values: \n{holidays.isna().sum()}")


holidays_by_type = holidays['type'].value_counts()
holidays_by_locale = holidays['locale'].value_counts()
holidays_per_year = holidays['year'].value_counts()

fig = make_subplots(rows = 3, cols = 1, subplot_titles = ('Total Holidays by Type', 'Total Holidays by Locale', 'Holidays per Year'))

fig.add_trace(
    go.Bar(x = holidays_by_type.index, y = holidays_by_type.values, text = holidays_by_type.values),
    row = 1, col = 1 
)

fig.add_trace(
    go.Bar(x = holidays_by_locale.index, y = holidays_by_locale.values, text = holidays_by_locale.values),
    row = 2, col = 1 
)

fig.add_trace(
    go.Bar(x = holidays_per_year.index, y = holidays_per_year.values, text = holidays_per_year.values),
    row = 3, col = 1 
)

fig.update_layout(height=1200, showlegend=False,
                  xaxis_title='Type',yaxis_title='Total Holidays',
                  xaxis2_title='jurisdiction',yaxis2_title='Total Holidays',
                  xaxis3_title='Year',yaxis3_title='Total Holidays',
                  )

fig.show()

In [ ]:
#Transactions table
transactions = pd.read_sql("SELECT * FROM transactions;",engine)

print("\nTransactions Summary:")
print(f"\nActive Stores: {transactions['store_nbr'].nunique()}")
print(f"\nMissing Values: \n{transactions.isna().sum()}")

query = """ 
        SELECT 
            EXTRACT(YEAR FROM date) as year,
            SUM(transactions) as total_transactions      
        FROM transactions
        GROUP BY year
        ORDER BY year;
        """

year_transactions = pd.read_sql(query,engine)

fig = px.bar(year_transactions,x ='year', y ='total_transactions',text='total_transactions',title = 'Total Transactions per Year')
fig.update_layout(separators=",.", xaxis_title='Year', yaxis_title='Total Transactions')
fig.update_traces(texttemplate='%{text:,.0f}')
fig.show()


In [ ]:
#Train

query = """
        SELECT * FROM train 
        ORDER BY DATE DESC
        LIMIT 100000;
         """

train_sample = pd.read_sql(query,engine)
print("\nSales sample: ")
display(train_sample.head())

print("\nSales sample stadistics: ")
print(f"\n{train_sample['unit_sales'].describe()}")

In [ ]:
#Sales info

date_query = """
            SELECT 
                MIN(date) as start_date,
                MAX(date) as end_date,
                MAX(date) - MIN(date) as date_span
            FROM train;

            """

date_range = pd.read_sql(date_query,engine)

print("Sales date range:")
print(f"\nDate spans from {date_range['start_date'].iloc[0]} to {date_range['end_date'].iloc[0]}")
print(f"Total days: {date_range['date_span'].iloc[0]}")



In [ ]:
# Refunds and null values on Train
query=""" 
        SELECT
            SUM(CASE WHEN unit_sales IS NULL THEN 1 ELSE 0 END) AS null_values,
            SUM(CASE WHEN unit_sales < 1 THEN 1 ELSE 0 END) AS refunds,
            SUM(CASE WHEN store_nbr IS NULL THEN 1 ELSE 0 END) AS null_stores,
            SUM(CASE WHEN item_nbr IS NULL THEN 1 ELSE 0 END) AS null_items
        FROM train  
"""

invalid_values = pd.read_sql(query,engine)

print("\nInvalid values on train: ")
print(f"Total refunds: {invalid_values['refunds'].iloc[0]}")
print(f"Null values: {invalid_values['null_values'].iloc[0]}")
print(f"Null stores registers: {invalid_values['null_stores'].iloc[0]}")
print(f"Null items registers: {invalid_values['null_items'].iloc[0]}")





Initial Findings:

1. Data Structure:
- 125M+ sales transactions.
- 54 stores across Ecuador.
- 4.100 unique product items.
- Data spans from 2013 to 2017.

2. Key observations:
    Stores:
    - Pichincha(19) and Guayas(11) are the states with the hightes store density.
    - Quito and Quayas have the largest number of stores.
    - Type 'D' is the most common store type, while 'E' is the least common.

    Items:
    - Grocery I, Beverages, and Produce have the largest assortment.
    - High ratio of perishable products (approx. 3:1 vs. non-perishable).
    
    Oil:
    - 38 explicit missing values.
    - ~400 are missing compared to the training timeline (likely weekends/holidays).

    Holidays:
    - 2016 has the hightest density of holiday events.
    - The dataset is dominated by National and Local holidays.

    Transactions:
    - All stores show active transaction activity.
    - Transaction volume remains relatively stable year-over-year.

    Train:
    - +500.000 refunds operations